# 04: Evaluation
Confusion matrices and per-technique F1 scores

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, os
from sklearn.metrics import confusion_matrix, classification_report, f1_score
print('=== NOTEBOOK 04: EVALUATION ===')
predictions = pd.read_csv('../results/predictions.csv')
label_map = pd.read_csv('../data/processed/label_map.csv')
actual_classes = sorted(predictions['y_true'].unique())
class_names = [label_map.loc[label_map['encoded'] == i, 'technique'].values[0] for i in actual_classes]
model_cols = [c for c in predictions.columns if c != 'y_true']
os.makedirs('../results/figures', exist_ok=True)
n_models = len(model_cols)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()
for idx, col in enumerate(model_cols):
    cm = confusion_matrix(predictions['y_true'], predictions[col], labels=actual_classes)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], xticklabels=class_names, yticklabels=class_names, cbar=False, square=True, linewidths=0.5)
    axes[idx].set_title(f'{col}'); axes[idx].set_xlabel('Predicted'); axes[idx].set_ylabel('True')
if n_models < 6: axes[5].axis('off')
plt.suptitle('Confusion Matrices', fontsize=15, y=1.02)
plt.tight_layout(); plt.savefig('../results/figures/02_confusion_matrices.png', dpi=300); plt.show()
print('Saved: 02_confusion_matrices.png')
f1_data = []
for col in model_cols:
    f1s = f1_score(predictions['y_true'], predictions[col], labels=actual_classes, average=None, zero_division=0)
    f1_data.append(f1s)
f1_matrix = np.array(f1_data)
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(f1_matrix, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1, xticklabels=class_names, yticklabels=model_cols, ax=ax, linewidths=0.5, linecolor='white', cbar_kws={'label': 'F1'})
ax.set_title('Per-Technique F1 Score'); ax.set_xlabel('ATT&CK Technique')
plt.tight_layout(); plt.savefig('../results/figures/04_f1_summary.png', dpi=300); plt.show()
print('Saved: 04_f1_summary.png')
print('\n=== DETAILED REPORTS ===')
for col in model_cols:
    print(f'\n--- {col} ---')
    print(classification_report(predictions['y_true'], predictions[col], labels=actual_classes, target_names=class_names, zero_division=0))
print('\n=== EVALUATION COMPLETE ===')
